# A* vs. Dijkstra's Algorithm: Side-by-Side Performance Comparison

In global path planning for autonomous robots, choosing the right search algorithm is crucial for performance. This notebook presents a comprehensive comparison between **Dijkstra's Algorithm** and the **A* (A-Star) Algorithm** in a 2D grid map with an inflation costmap (representing a realistic ROS 2 Nav2 environment).

## 1. Why A* is More Efficient: The Mathematical Reason

Both algorithms find the optimal (shortest) path, but they explore the space very differently:

| Algorithm | Evaluation Function $f(n)$ | Heuristic $h(n)$ | Search Behavior |
| :--- | :--- | :--- | :--- |
| **Dijkstra** | $f(n) = g(n)$ | $h(n) = 0$ | **Isotropic (Radial)** - Explores in all directions equally like waves in water. |
| **A\*** | $f(n) = g(n) + h(n)$ | $h(n) = \text{Euclidean distance to goal}$ | **Goal-Oriented (Focused)** - Strongly biased towards the goal. |

Where:
- $g(n)$: The exact cost accumulated from the start node to the current node $n$.
- $h(n)$: The estimated (heuristic) remaining cost from node $n$ to the goal. In 2D grids, this is typically the straight-line **Euclidean distance**.

---

## 2. The Code-Level Difference (The Magic of the Priority Queue)

In the code, the priority queue (implemented via Python's `heapq`) sorts nodes based on the first element of the pushed tuple:

### Dijkstra's Pushing Mechanism:
```python
heapq.heappush(open_list, (new_g, nx, ny))
```
- The queue is sorted **only by the accumulated cost `new_g`**.
- Result: The algorithm expands the cell that is closest to the **start point**, leading to a round, expanding wavefront that wastes time exploring areas in the opposite direction of the goal.

### A* Pushing Mechanism:
```python
h_n = np.hypot((gx - nx)*RES, (gy - ny)*RES)
f_n = new_g + h_n
heapq.heappush(open_list, (f_n, new_g, nx, ny))
```
- The queue is sorted by **`f_n = new_g + h_n`**.
- Result: A cell that lies in the direction of the goal will have a much smaller $h(n)$ value. Even if its $g(n)$ is slightly higher, its total $f(n)$ will be smaller than a cell going in the wrong direction. Therefore, the priority queue pops goal-ward cells first, focusing the search beam directly towards the target!

## Step 1 — Setup and Optimized Map Generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import distance_transform_edt
import heapq
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Map configuration
GRID_W, GRID_H = 100, 100
RES = 0.1
MAP_W, MAP_H = GRID_W * RES, GRID_H * RES

START_M = (1.0, 1.0)
GOAL_M = (9.0, 9.0)

# Obstacles
OBSTACLES_RECT = [
    (0.0, 0.0, 10.0, 0.2), (0.0, 9.8, 10.0, 0.2),
    (0.0, 0.0, 0.2, 10.0), (9.8, 0.0, 0.2, 10.0),
    (3.0, 0.0, 0.2, 5.5), (6.0, 4.5, 0.2, 5.5),
    (1.5, 3.0, 3.0, 0.2), (5.5, 6.0, 3.5, 0.2),
    (0.0, 7.0, 2.5, 0.2), (7.5, 2.0, 2.5, 0.2),
    (4.0, 4.0, 1.5, 1.5), (1.0, 5.5, 1.0, 1.0),
    (8.0, 7.5, 1.0, 1.0),
]

OBSTACLES_CIRCLE = [
    {'cx': 2.5, 'cy': 1.5, 'r': 0.6},
    {'cx': 7.0, 'cy': 5.0, 'r': 0.7},
    {'cx': 4.5, 'cy': 8.0, 'r': 0.5},
    {'cx': 8.5, 'cy': 3.0, 'r': 0.4},
]

def build_occupancy_grid():
    # Vectorized grid generation (Super Fast)
    x_coords = (np.arange(GRID_W) + 0.5) * RES
    y_coords = (np.arange(GRID_H) + 0.5) * RES
    X, Y = np.meshgrid(x_coords, y_coords)
    
    grid = np.zeros((GRID_H, GRID_W), dtype=np.int8)
    occ_mask = np.zeros((GRID_H, GRID_W), dtype=bool)
    
    for rx, ry, rw, rh in OBSTACLES_RECT:
        occ_mask |= (rx <= X) & (X <= rx + rw) & (ry <= Y) & (Y <= ry + rh)
        
    for c in OBSTACLES_CIRCLE:
        occ_mask |= (X - c['cx'])**2 + (Y - c['cy'])**2 <= c['r']**2
        
    grid[occ_mask] = 100
    return grid

ROBOT_RADIUS_M = 0.25
INFLATION_RADIUS_M = 0.5
LETHAL_COST = 254
INSCRIBED_COST = 253

def compute_inflation_costmap(occ_grid, robot_r_m, inflation_r_m):
    h, w = occ_grid.shape
    # Ultra-fast Euclidean Distance Transform
    dist_m = distance_transform_edt(occ_grid < 100) * RES
    
    cost = np.zeros((h, w), dtype=np.uint8)
    lethal_mask = (occ_grid >= 100)
    inscribed_mask = (~lethal_mask) & (dist_m <= robot_r_m)
    inflation_mask = (~lethal_mask) & (dist_m > robot_r_m) & (dist_m <= robot_r_m + inflation_r_m)
    
    cost[lethal_mask] = LETHAL_COST
    cost[inscribed_mask] = INSCRIBED_COST
    
    factor = (dist_m[inflation_mask] - robot_r_m) / inflation_r_m
    c = (INSCRIBED_COST * np.exp(-5.0 * factor)).astype(np.int32)
    c = np.clip(c, 1, INSCRIBED_COST)
    cost[inflation_mask] = c
    
    return cost, dist_m

occupancy_grid = build_occupancy_grid()
costmap, _ = compute_inflation_costmap(occupancy_grid, ROBOT_RADIUS_M, INFLATION_RADIUS_M)
print("✅ Grid map and inflation costmap successfully built using vectorized methods!")

## Step 2 — Implementing Both Path Planners

In [ ]:
def m_to_idx(x_m, y_m):
    ix = int(np.clip(np.floor(x_m / RES), 0, GRID_W - 1))
    iy = int(np.clip(np.floor(y_m / RES), 0, GRID_H - 1))
    return ix, iy

NEIGHBORS = [
    (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
    (-1, -1, np.sqrt(2)), (-1, 1, np.sqrt(2)),
    (1, -1, np.sqrt(2)), (1, 1, np.sqrt(2)),
]

def dijkstra(costmap, start_m, goal_m):
    sx, sy = m_to_idx(*start_m)
    gx, gy = m_to_idx(*goal_m)
    h, w = costmap.shape
    
    g_map = np.full((h, w), np.inf, dtype=np.float32)
    parent = np.full((h, w, 2), -1, dtype=np.int32)
    visited = np.zeros((h, w), dtype=np.bool_)
    search_order = []
    
    open_list = []
    g_map[sy, sx] = 0.0
    heapq.heappush(open_list, (0.0, sx, sy))
    order_idx = 0
    
    while open_list:
        g, cx, cy = heapq.heappop(open_list)
        
        if visited[cy, cx]:
            continue
        visited[cy, cx] = True
        search_order.append((cx, cy, order_idx))
        order_idx += 1
        
        if cx == gx and cy == gy:
            break
            
        for dx, dy, step_cost in NEIGHBORS:
            nx, ny = cx + dx, cy + dy
            if not (0 <= nx < w and 0 <= ny < h):
                continue
            if visited[ny, nx]:
                continue
            if costmap[ny, nx] >= LETHAL_COST:
                continue
                
            cell_cost = costmap[ny, nx] / 254.0
            move_cost = step_cost * RES * (1.0 + 5.0 * cell_cost)
            new_g = g + move_cost
            
            if new_g < g_map[ny, nx]:
                g_map[ny, nx] = new_g
                parent[ny, nx] = [cx, cy]
                heapq.heappush(open_list, (new_g, nx, ny))
                
    goal_reached = visited[gy, gx]
    return g_map, parent, visited, search_order, goal_reached

def astar(costmap, start_m, goal_m):
    sx, sy = m_to_idx(*start_m)
    gx, gy = m_to_idx(*goal_m)
    h, w = costmap.shape
    
    g_map = np.full((h, w), np.inf, dtype=np.float32)
    parent = np.full((h, w, 2), -1, dtype=np.int32)
    visited = np.zeros((h, w), dtype=np.bool_)
    search_order = []
    
    open_list = []
    g_map[sy, sx] = 0.0
    h0 = np.hypot((gx - sx)*RES, (gy - sy)*RES)
    heapq.heappush(open_list, (h0, 0.0, sx, sy))
    order_idx = 0
    
    while open_list:
        f, g, cx, cy = heapq.heappop(open_list)
        
        if visited[cy, cx]:
            continue
        visited[cy, cx] = True
        search_order.append((cx, cy, order_idx))
        order_idx += 1
        
        if cx == gx and cy == gy:
            break
            
        for dx, dy, step_cost in NEIGHBORS:
            nx, ny = cx + dx, cy + dy
            if not (0 <= nx < w and 0 <= ny < h):
                continue
            if visited[ny, nx]:
                continue
            if costmap[ny, nx] >= LETHAL_COST:
                continue
                
            cell_cost = costmap[ny, nx] / 254.0
            move_cost = step_cost * RES * (1.0 + 5.0 * cell_cost)
            new_g = g + move_cost
            
            if new_g < g_map[ny, nx]:
                g_map[ny, nx] = new_g
                parent[ny, nx] = [cx, cy]
                h_n = np.hypot((gx - nx)*RES, (gy - ny)*RES)
                f_n = new_g + h_n
                heapq.heappush(open_list, (f_n, new_g, nx, ny))
                
    goal_reached = visited[gy, gx]
    return g_map, parent, visited, search_order, goal_reached

## Step 3 — Performance Comparison and Benchmarking

Let's run both planners on the exact same map and record their execution time and the total number of expanded nodes.

In [ ]:
print("🔄 Running Dijkstra's Algorithm...")
t_dijk_start = time.perf_counter()
g_dijk, parent_dijk, visited_dijk, search_dijk, ok_dijk = dijkstra(costmap, START_M, GOAL_M)
t_dijk_end = time.perf_counter()
elapsed_dijk = (t_dijk_end - t_dijk_start) * 1000  # ms
nodes_dijk = visited_dijk.sum()

print("🔄 Running A* Algorithm...")
t_astar_start = time.perf_counter()
g_astar, parent_astar, visited_astar, search_astar, ok_astar = astar(costmap, START_M, GOAL_M)
t_astar_end = time.perf_counter()
elapsed_astar = (t_astar_end - t_astar_start) * 1000  # ms
nodes_astar = visited_astar.sum()

print("\n================ BENCHMARK RESULT ================")
print(f"Dijkstra: Time = {elapsed_dijk:.2f} ms | Explored Cells = {nodes_dijk}")
print(f"A*      : Time = {elapsed_astar:.2f} ms | Explored Cells = {nodes_astar}")
print(f"--> A* explored {nodes_dijk - nodes_astar} fewer cells ({((nodes_dijk-nodes_astar)/nodes_dijk)*100:.1f}% reduction!)")
print(f"--> A* speedup factor: {elapsed_dijk / elapsed_astar:.2f}x faster!")
print("==================================================")

## Step 4 — Visualizing the Wavefront Expansion (Side-by-Side)

This visualization highlights the difference in the explored area (marked in blue). Notice how Dijkstra explores almost the entire map radially, whereas A* concentrates its search directly in the direction of the goal.

In [ ]:
def reconstruct_path(parent, gx, gy):
    path = []
    cx, cy = gx, gy
    if parent[cy, cx, 0] == -1:
        return []
    while cx != -1 and cy != -1:
        path.append((cx, cy))
        px, py = parent[cy, cx]
        if px == cx and py == cy:
            break
        cx, cy = px, py
    return path[::-1]

gx, gy = m_to_idx(*GOAL_M)
path_dijk = reconstruct_path(parent_dijk, gx, gy)
path_astar = reconstruct_path(parent_astar, gx, gy)

cmap_cost = LinearSegmentedColormap.from_list(
    'nav2_cost', ['#000000', '#ff0000', '#ffff00', '#00aaff', '#ffffff'])

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Plot Dijkstra
axes[0].imshow(costmap, origin='lower', cmap=cmap_cost,
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=254, alpha=0.35)
xs_d = [p[0] for p in search_dijk]
ys_d = [p[1] for p in search_dijk]
axes[0].scatter(np.array(xs_d)*RES + RES/2, np.array(ys_d)*RES + RES/2,
               c='dodgerblue', s=6, alpha=0.7, label='Explored Area')
if path_dijk:
    px_d = np.array([p[0] for p in path_dijk]) * RES + RES/2
    py_d = np.array([p[1] for p in path_dijk]) * RES + RES/2
    axes[0].plot(px_d, py_d, 'g-', linewidth=3.0, label='Shortest Path')
axes[0].scatter(*START_M, c='lime', s=180, marker='o', edgecolors='black', zorder=5, label='Start')
axes[0].scatter(*GOAL_M, c='red', s=180, marker='X', edgecolors='black', zorder=5, label='Goal')
axes[0].set_title(f"Dijkstra Wavefront Expansion\nExplored Nodes: {nodes_dijk}", fontsize=12)
axes[0].set_xlim(0, MAP_W); axes[0].set_ylim(0, MAP_H)
axes[0].set_aspect('equal')
axes[0].legend(loc='upper right', fontsize=9)

# Plot A*
axes[1].imshow(costmap, origin='lower', cmap=cmap_cost,
              extent=[0, MAP_W, 0, MAP_H], vmin=0, vmax=254, alpha=0.35)
xs_a = [p[0] for p in search_astar]
ys_a = [p[1] for p in search_astar]
axes[1].scatter(np.array(xs_a)*RES + RES/2, np.array(ys_a)*RES + RES/2,
               c='dodgerblue', s=6, alpha=0.7, label='Explored Area')
if path_astar:
    px_a = np.array([p[0] for p in path_astar]) * RES + RES/2
    py_a = np.array([p[1] for p in path_astar]) * RES + RES/2
    axes[1].plot(px_a, py_a, 'g-', linewidth=3.0, label='Shortest Path')
axes[1].scatter(*START_M, c='lime', s=180, marker='o', edgecolors='black', zorder=5, label='Start')
axes[1].scatter(*GOAL_M, c='red', s=180, marker='X', edgecolors='black', zorder=5, label='Goal')
axes[1].set_title(f"A* Goal-Oriented Expansion\nExplored Nodes: {nodes_astar}", fontsize=12)
axes[1].set_xlim(0, MAP_W); axes[1].set_ylim(0, MAP_H)
axes[1].set_aspect('equal')
axes[1].legend(loc='upper right', fontsize=9)

plt.suptitle("Dijkstra vs. A* Search Area Expansion Comparison", fontsize=15, y=0.98)
plt.tight_layout()
plt.show()

## 3. Key Takeaways

1. **Identical Paths**: Notice that the green path generated by both algorithms is **exactly the same**. A* does not sacrifice path optimality; it achieves the exact same optimal path with much less computation.
2. **Huge Search Space Reduction**: In the A* visual plot, the explored area is highly concentrated towards the goal, bypassing hundreds of cells behind or far side of the start point that Dijkstra wasted time exploring.
3. **Real-world Significance**: In ROS 2 Nav2, the default global planner (`NavFn`) uses Dijkstra, but can be switched to A* (`SmacPlanner`). On large, high-resolution robot maps (e.g. $1000 \times 1000$), A* reduces global path computation time from seconds to milliseconds, which is critical for real-time collision avoidance and dynamic replanning!